### Webscraper for extracting newly published names, strains and accession numbers from the weeekly IJSEM email (saved in html). Script then compares the IJSEM names to the NCBI names and generates a report used for taxonomy updates. 
### Uses Natural Language processing combined with REGEX to extract information from text
### Notes on selenium. There is a lot of bad advise in stack overflow that refers to old versions of selenium. Need to refer to the offocial documentation https://selenium-python.readthedocs.io/ If you run out of space in your linux directory the script will crash. There is a cache file written by selenium that may then need to be deleted to get the chrome driver working again. Follow the path in the error message and delete that cache directory. Selenium needs a webdriver installed, find details at https://sites.google.com/chromium.org/driver/downloads

### NOTE selenium cache files are quite large and need to be cleaned up periodically

In [ ]:
pip install "numpy<2.0" 

In [ ]:
pip install spacy
#pip install -U pip setuptools wheel
#pip install -U spacy
#pip install --upgrade spacy

### Install the model at the command lines
small model
python -m spacy download en_core_web_sm

medium model
python -m spacy download en_core_web_md

large model
python -m spacy download en_core_web_lg

### create training set https://spacy.io/usage/training

Label-studio works the best for labelling data. Export as .json when finished, they use labelstudio_to_spacy2.py to convert the .json files to .spacy file for training. 

https://spacy.io/usage/training#quickstart

python -m spacy init fill-config ./base_config.cfg ./config.cfg

This NER annotator worked better than the one above. Used the DocBin technique to convert to .spacy file. Spacy training command with config.cfg then worked.

Config.cfg needs to be created for the spacy command line training. 

Follow the instructions in the quickstart for the base_config.cfg file. Selected OS and clicked NER

https://spacy.io/usage/training#quickstart

Run the following at command line to create the config.cfg. 

python -m spacy init fill-config ./base_config.cfg ./config.cfg

Once the config file was done and spacy file was created run the following at command line to train the model

To debug the data:
python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./train.spacy

To debug the config file file:
python -m spacy train config.cfg --output ./output

or

python -m spacy train config.cfg 

train the model at command line
python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./train.spacy

creates two directories model-best and model-last. I used model-best 

nlp1 = spacy.load(r"./output/model-best")


Notes
python -m spacy init fill-config ./base_config.cfg ./config.cfg

python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy

https://spacy.io/usage/training#basics

### Train spacy if not already done. Using https://labelstud.io/


In [ ]:
from spacy.tokens import DocBin
import pandas as pd
import json
import os

### train the model at command line
python -m spacy train config3.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy 

### Main webscraper program start


In [1]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import time
import IPython
import pandas as pd
import re
import os
import sys
import bs4
from bs4 import BeautifulSoup
#import requests
import numpy as np
#from datetime import datetime
from nameparser import HumanName
import spacy
#nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("en_core_web_md")
from spacy.matcher import PhraseMatcher
from spacy import displacy

In [2]:
# --- Baseline 1.6 changes: load trained NER model once + organism NER + debug helpers ---
# Trained entity labels: "strain", "accession", "organism", "basionym"

nlp_strain = spacy.load(r"./output/model-best")  # trained model (shared)

ALLOWED_STRAIN_LABELS = {"strain"}
ALLOWED_ORGANISM_LABELS = {"organism"}
ALLOWED_BASIONYM_LABELS = {"basionym"}

# PhraseMatcher used to gate strain extraction to sentences mentioning strain keywords
phrase_matcher = PhraseMatcher(nlp.vocab)
strain_keywords = ["strain", "Strain", "strains", "Strains"]
patterns = [nlp.make_doc(p) for p in strain_keywords]
phrase_matcher.add("STRAIN_CTX", patterns)

def _clean_ascii(s: str) -> str:
    return s.encode("ascii", "ignore").decode("utf-8", errors="ignore").strip()

def find_strains(description: str):
    """Return only 'strain' entities from the trained model (sentence-gated)."""
    if not description:
        return []
    results = []
    doc = nlp(description)
    for sent in doc.sents:
        if not phrase_matcher(nlp(sent.text)):
            continue
        doc2 = nlp_strain(sent.text)
        for ent in doc2.ents:
            if ent.label_ in ALLOWED_STRAIN_LABELS:
                val = _clean_ascii(ent.text.strip())
                if val and val not in results:
                    results.append(val)
    return results

def find_organisms(description: str):
    """Return only 'organism' entities from the trained model (no gating)."""
    if not description:
        return []
    results = []
    doc2 = nlp_strain(description)
    for ent in doc2.ents:
        if ent.label_ in ALLOWED_ORGANISM_LABELS:
            val = _clean_ascii(ent.text.strip())
            if val and val not in results:
                results.append(val)
    return results

def find_basionyms(description):
    """
    Find basionym entities using the trained spaCy NER model.
    Returns only entities labeled 'basionym' (de-duplicated).
    """
    if not description:
        return []

    results = []
    doc2 = nlp_strain(description)
    for ent in doc2.ents:
        if ent.label_ in ALLOWED_BASIONYM_LABELS:
            val = ent.text.strip()
            val = val.encode("ascii", "ignore").decode("utf-8", errors="ignore").strip()
            if val and val not in results:
                results.append(val)
    return results

# Debug controls
DEBUG_NER = False
DEBUG_URL_LIMIT = 200          # only debug first N URLs encountered
DEBUG_DESC_PER_URL = 200       # debug only first N descriptions per URL

_debug_seen_urls = set()
_debug_desc_count = {}

from IPython.display import display, HTML

def debug_ner(description: str, *, url: str = "", header: str = ""):
    """Visualize entities from the trained model and show extracted lists."""
    if not DEBUG_NER:
        return
    if url and url not in _debug_seen_urls and len(_debug_seen_urls) >= DEBUG_URL_LIMIT:
        return

    if url:
        _debug_desc_count.setdefault(url, 0)
        if _debug_desc_count[url] >= DEBUG_DESC_PER_URL:
            return
        _debug_desc_count[url] += 1
        _debug_seen_urls.add(url)

    orgs = find_organisms(description)
    strains = find_strains(description)

    doc2 = nlp_strain(description)

    title_html = f"<h3>{header}</h3>" if header else ""
    url_html = f"<p><b>URL:</b> {url}</p>" if url else ""
    lists_html = (
        "<p><b>Extracted (NER):</b><br>"
        f"<b>organism</b>: {orgs}<br>"
        f"<b>strain</b>: {strains}</p>"
    )

    # Show a short snippet of the description to orient debugging
    snippet = (description[:1500] + "…") if len(description) > 1500 else description
    snippet_html = f"<details><summary><b>Description text (click to expand)</b></summary><pre>{snippet}</pre></details>"

    display(HTML(title_html + url_html + lists_html + snippet_html))
    # Render entities inline
    display(HTML(displacy.render(doc2, style="ent")))

def remove_non_ascii(text):
    """Remove non-ASCII characters"""
    return ''.join(char for char in text if ord(char) < 128)

In [3]:
#test block for testing displacy
text2 = ("The type strain is 2205BS29-5T (=LMG 33062T =KACC 23240T), which is isolated from a marine sponge, P. elegans, "
"collected from Beomseom in Jeju-Island, Republic of Korea."
"The DNA G+C content of strain 2205BS29-5T is 67.8%. The GenBank accession numbers for the 16S rRNA gene and" 
"whole-genome sequences of strain 2205BS29-5T are OQ569368 and JAVAMQ000000000, respectively." 
"Basionym: Candida aaseri Dietrichson ex Uden and H.R. Buckley, The Yeasts: a Taxonomic Study, 912(1970)")

In [4]:
#test but need this later
import spacy
from spacy import displacy
from IPython.display import display

nlp1 = spacy.load("./output/model-best")
doc = nlp1(text2)

display(displacy.render(doc, style="ent", jupyter=True))

None

### Find URLS from saved email in html - save as from outlook in htm format. URLs are extracted and saved as input for selenium

In [5]:
input_file = (r'IJSEMemail93.htm')
output = (r'NameCheckweek92.xlsx')
alldescriptions = (r'all_descriptions92')

In [4]:
#Only need this if you suspect a problem with beautiful soup not imporing the URL list correctly. Otherwise skip this step
from bs4.diagnose import diagnose
with open (input_file, encoding = 'unicode_escape') as f:
    data = f.read()
diagnose(data)

Diagnostic running on Beautiful Soup 4.14.3
Python version 3.13.13 (main, Apr 14 2026, 10:28:59) [GCC 8.5.0 20210514 (Red Hat 8.5.0-22)]
I noticed that html5lib is not installed. Installing it may help.
Found lxml version 6.1.0.0
Trying to parse your markup with html.parser
Here's what html.parser did with the markup:
<html xmlns="http://www.w3.org/TR/REC-html40" xmlns:m="http://schemas.microsoft.com/office/2004/12/omml" xmlns:ns0="http://www.w3.org/1999/xhtml" xmlns:o="urn:schemas-microsoft-com:office:office" xmlns:v="urn:schemas-microsoft-com:vml" xmlns:w="urn:schemas-microsoft-com:office:word">
 <head>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="Word.Document" name="ProgId"/>
  <meta content="Microsoft Word 15" name="Generator"/>
  <meta content="Microsoft Word 15" name="Originator"/>
  <link href="IJSEMemail91_files/filelist.xml" rel="File-List"/>
  <link href="IJSEMemail91_files/editdata.mso" rel="Edit-Time-Data"/>
  <!--[if !mso]>
<styl

#### alternative Beautifiul soup code for extracting URLS with autodetect encoding

In [6]:
#alternative Beautifiul soup with autodetect encoding
from charset_normalizer import from_path

# Auto-detect file encoding
result = from_path(input_file).best()
html = str(result)

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
text = soup.get_text()

# Extract all http/https links
urls = re.findall(r"https?://\S+", text)
urls = [url.rstrip('.,);') for url in urls]

# Filter links as needed
filtered_urls = [
    url for url in urls
    if all(exclude not in url for exclude in ["TandC", "myaccount", "join-the-society"])
    # Remove "doi.org" from filter if you want article links
]

if not filtered_urls:
    raise ValueError("No usable URLs found!")

print(f"Found {len(filtered_urls)} URLs:")
print("\n".join(filtered_urls))


Found 10 URLs:
http://dx.doi.org/10.1099/ijsem.0.007171
http://dx.doi.org/10.1099/ijsem.0.007163
http://dx.doi.org/10.1099/ijsem.0.007175
http://dx.doi.org/10.1099/ijsem.0.007107
http://dx.doi.org/10.1099/ijsem.0.007167
http://dx.doi.org/10.1099/ijsem.0.007164
http://dx.doi.org/10.1099/ijsem.0.007159
http://dx.doi.org/10.1099/ijsem.0.007172
http://dx.doi.org/10.1099/ijsem.0.007166
http://dx.doi.org/10.1099/ijsem.0.007165


In [ ]:
# (Moved strain/organism NER functions above for clarity; this cell is now unused in the debug notebook.)

### Main body - Selenium to extract data from html
### https://nameparser.readthedocs.io/en/latest/index.html to find the last name

In [7]:
import tempfile
pub_df = pd.DataFrame(columns=['PublishedName', 'Accessions', 'Strains', 'Basionym',  'Authority', 'DOI', 'filtered_url'])
pd.set_option('display.max_columns', None)
combined_description = []

for filtered_url in filtered_urls:
    temp_profile = tempfile.mkdtemp()

    options = Options()
    #options.binary_location = chrome_path
    #options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)

    counter = 1
    strains = []
    accessions = []
    orgname = []
    doi = []
    author = []
    date = []
    year = []
    author_raw = []
    author = []
    name1 = []
    name2 = []
    author_count = []
    authority = []
    description = None
    description1 = []
    description2 = []
    snumber = []
    basionym = []

    #Navigate to the webpage
    driver.get(filtered_url)

    #Allow time for dynamic content to load (you may need to use WebDriverWait for more robust waiting)
    time.sleep(5)

    html = driver.page_source
    
    for element in driver.find_elements(By.CLASS_NAME, "item-meta-data__item-title"):
        #print(element.text)
        title = element.text
        print(title)
        
    for element in driver.find_elements(By.PARTIAL_LINK_TEXT, "doi.org"):
        doi = element.text
        print(doi)
    
    #find publication year
    for element in driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[3]/span/span[2]"):
        date = element.text
        year = date[-4:]
        #print(year)
    
    #find authors
    for element in driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[1]/span"):
        author_raw = element.text
        #print(author_raw)
        author = re.sub(r"[\d+]+",'',author_raw)
        author = re.sub(r"†",'',author)
        author = re.sub(r" and ",',',author)
        author = re.sub(r",,",',',author)  
        author = author.split(',')
        author[:] = [item for item in author if item != '']
        #print(author)
        author_count = len(author)
        #print(author_count)
        if author_count == 1:
            name1 = (str(author[0])) 
            name1 = HumanName(name1)
            authority = name1.last + ' ' +str(year)
            #print(authority)
        elif author_count ==2:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            name2 = (str(author[1]))
            name2 = HumanName(name2)
            authority = name1.last + ' and ' + name2.last + ' ' +str(year)
            #print(authority)
        else:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            authority = name1.last + ' et al. ' +str(year)
            #print(authority)
    
    #extract data from each species description
    for element in driver.find_elements(By.CSS_SELECTOR, "div.tl-main-part.title"): #finds section headers
        #print(element.text)
        counter += 1
        description = element.text
        print(description)
        if "Description of" in description:
            strains = []
            print('found', description)  
            #snumber = 's' + str(counter - 4) + '/p[3]'
            snumber = 's' + str(counter - 4)
            print('snumber is', snumber)
            for element in driver.find_elements(By.ID, snumber):
                description = element.text
                cleaned_text = remove_non_ascii(description)
                combined_description.append(cleaned_text)
                debug_ner(cleaned_text, url=filtered_url, header='Description block (debug)')
                #print(cleaned_text)
                print(description)
                        
                # find the organism names (NER - label 'organism')
                orgname = find_organisms(cleaned_text)

                #find the accessions
                pattern = [r'[A-Z]{2}\d{6}', r'[A-Z]{4}\d{8}', r'([A-Z]+)(_[A-Z]+)\d{6}', r'[A-Z]{6}\d{9}']
                regex = re.compile(r'\b(' + '|'.join(pattern) + r')\b')
                if description is not None:
                    accessions = [m.group() for m in regex.finditer(description)]
                    print('accessions', accessions)
    
                #find the strains
                if description is not None:
                    strains = find_strains(cleaned_text)
                    print('strain names', strains)

                #find the basionyms
                basionym = find_basionyms(cleaned_text) if description is not None else []
    
                #load data into pandas dataframe
                row_data = [orgname, accessions, strains, basionym, authority, doi, filtered_url]
                length = len(pub_df)
                pub_df.loc[length] = row_data
            print('BREAK1')

   
    for element in driver.find_elements(By.CLASS_NAME, "tl-lowest-section"): #finds section headers
        description1 = element.text
        outer_html = element.get_attribute("outerHTML")
        if "Description of" in description1:          
            print(outer_html)
            spans = soup.find_all('span', attrs = {'class' : 'tl-lowest-section'})
            for span in spans:
                if "Description of" in span.text:   
                    #print (span.text)
                    outer_div_id = span.find_parent('div').get('id')
                    #print(f"Outer div ID: {outer_div_id}, Text: {span.text}")
                    for element in driver.find_elements(By.ID, outer_div_id):
                        description = element.text
                        cleaned_text = remove_non_ascii(description)
                        combined_description.append(cleaned_text)
                    debug_ner(cleaned_text, url=filtered_url, header='Description block (debug)')
                        #print(cleaned_text)
                    print(description)
                        
                    # find the organism names (NER - label 'organism')
                    orgname = find_organisms(cleaned_text)

                    #find the accessions
                    pattern = [r'[A-Z]{2}\d{6}', r'[A-Z]{4}\d{8}', r'([A-Z]+)(_[A-Z]+)\d{6}', r'[A-Z]{6}\d{9}']
                    regex = re.compile(r'\b(' + '|'.join(pattern) + r')\b')
                    if description is not None:
                        accessions = [m.group() for m in regex.finditer(description)]
                        print('accessions', accessions)
    
                    #find the strains
                    strains = []
                    if description is not None:
                        strains = find_strains(cleaned_text)
                        print('strain names', strains)

                    #find the basionyms
                    basionym = find_basionyms(cleaned_text) if description is not None else ""
  
                    basionym_list = find_basionyms(cleaned_text)
                    basionym_for_row = ", ".join(basionym_list) if basionym_list else ""
    
                    #load data into pandas dataframe
                    row_data = [orgname, accessions, strains, basionym, authority, doi, filtered_url]
                    length = len(pub_df)
                    pub_df.loc[length] = row_data
                    print('BREAK2')

        
#Close the browser window
    driver.quit()   
print("Scraper Done")

Clavibacter vidaverae sp. nov., Clavibacter rahimiani sp. nov. and Clavibacter davisi sp. nov.: corynebacterial plant pathogens isolated from small-grain cereals
https://doi.org/10.1099/ijsem.0.007154
Streptomyces mabaensis sp. nov., isolated from karst cave samples of ancient Ma'ba remains
https://doi.org/10.1099/ijsem.0.007153
Formosa sejongensis sp. nov. and Polaromonas potterensis sp. nov., isolated from King George Island, Antarctica
https://doi.org/10.1099/ijsem.0.007150
Paucilactobacillus salsurae sp. nov. and Enterococcus salsurae sp. nov., isolated from traditional Chinese pickle
https://doi.org/10.1099/ijsem.0.007152
A comprehensive genome-centred taxonomy for agrobacteria and rhizobia in the Bartonellaceae and Rhizobiaceae families
https://doi.org/10.1099/ijsem.0.007125
Thermus javaensis sp. nov., a novel thermophilic bacterium isolated from litter of a geyser in Cisolok, West Java, Indonesia
https://doi.org/10.1099/ijsem.0.007136
Tongtianella jiaweipingae gen. nov., sp. nov

In [13]:
#optional write description to a file
#print(combined_description)
file = open(alldescriptions, "w")
file.writelines(combined_description)
file.close()

In [33]:
pub_df

,PublishedName,Accessions,Strains,Basionym,Authority,DOI,filtered_url


In [15]:
def non_empty_list(x):
    return isinstance(x, list) and len(x) > 0

pub_df = pub_df[
    pub_df["PublishedName"].apply(non_empty_list) &
    pub_df["Accessions"].apply(non_empty_list)
]

print("Rows after organism + accession filter:", pub_df.shape)

Rows after organism + accession filter: (2, 7)


In [16]:
pub_df

,PublishedName,Accessions,Strains,Basionym,Authority,DOI,filtered_url
0,[Formosa sejongensis],"[OR687713, JAWQUN000000000]","[PL04T, DSM 117045T, KCTC 102051T]",[],Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
2,[Polaromonas potterensis],"[OR742320, JAWQLX000000000]","[SM01, DSM 116566, KCTC 8096]",[],Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150


In [17]:
pd.set_option('max_colwidth', None)
pub_df = pub_df.copy()
pub_df.loc[:, 'Strains'] = pub_df['Strains'].apply(lambda l: ", ".join(map(str, l)) if isinstance(l, list) else "")
pub_df.loc[:, 'Strains'] = pub_df['Strains'].astype(pd.StringDtype())
pub_df.loc[:, 'Strains'] = pub_df['Strains'].str.replace(',', ', ')
pub_df["Basionym"] = pub_df["Basionym"].apply(lambda x: ", ".join(x) if isinstance(x, list) and len(x) > 0 else "")
pub_df.explode(['PublishedName']).reset_index(drop=True)

,PublishedName,Accessions,Strains,Basionym,Authority,DOI,filtered_url
0,Formosa sejongensis,"[OR687713, JAWQUN000000000]","PL04T, DSM 117045T, KCTC 102051T",,Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
1,Polaromonas potterensis,"[OR742320, JAWQLX000000000]","SM01, DSM 116566, KCTC 8096",,Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150


In [ ]:
#pub_df['Strains'] = pub_df['Strains'].astype(pd.StringDtype())
#pub_df['Strains'] = pub_df['Strains'].str.replace(',', ', ')
#pub_df

In [18]:
pub2_df = pub_df.explode(['Accessions']).reset_index(drop=True)
#pub2_df
pub4_df = pub2_df.explode(['PublishedName']).reset_index(drop=True)
pub4_df.rename(columns={'Accessions' : 'accession'}, inplace=True)
#pub4_df = pub4_df.dropna()
#pub4_df = pub4_df.drop_duplicates(subset='accession', keep="first")
pub4_df=pub4_df[pub4_df['accession'].isnull() | ~pub4_df[pub4_df['accession'].notnull()].duplicated(subset='accession',keep='first')]
#pub4_df

In [19]:
df_unique= pub4_df.drop_duplicates(["accession"], keep="first")
#df_unique = df_unique.dropna()
df_unique

,PublishedName,accession,Strains,Basionym,Authority,DOI,filtered_url
0,Formosa sejongensis,OR687713,"PL04T, DSM 117045T, KCTC 102051T",,Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
1,Formosa sejongensis,JAWQUN000000000,"PL04T, DSM 117045T, KCTC 102051T",,Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
2,Polaromonas potterensis,OR742320,"SM01, DSM 116566, KCTC 8096",,Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
3,Polaromonas potterensis,JAWQLX000000000,"SM01, DSM 116566, KCTC 8096",,Choi et al. 2026,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150


### Create a dataframe of unique accessions and look up NCBI taxonomy information of each accession with srcchk

In [ ]:
#df_unique.dtypes

In [20]:
#df_unique['accession'] = df_unique['accession'].astype('str') 
df_unique.loc[:, 'accession'] = df_unique['accession'].astype('str') 

In [21]:
with open('acclist', 'w') as f:
    for text in df_unique['accession'].tolist():
        f.write(text + '\n')

In [22]:
os.system("/netopt/ncbi_tools64/bin/srcchk -i acclist -f taxname,taxid,strain -o acclist.taxdata")


0

In [23]:
taxdata_file_name = (r'acclist.taxdata')    
srcchk_df = pd.read_csv(taxdata_file_name, sep='\t', index_col=None, low_memory=False)
srcchk_df.drop(columns=['Unnamed: 4'], inplace=True)
srcchk_df.rename(columns={'organism' : 'NCBIname'}, inplace=True)
srcchk_df['accession'] = srcchk_df['accession'].astype(str).replace('\.\d+', '', regex=True).astype(str)
srcchk_df = srcchk_df.dropna(subset=['NCBIname'])
srcchk_df 

<>:5: SyntaxWarning: invalid escape sequence '\.'
<>:5: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_238669/4252064212.py:5: SyntaxWarning: invalid escape sequence '\.'
  srcchk_df['accession'] = srcchk_df['accession'].astype(str).replace('\.\d+', '', regex=True).astype(str)


,accession,NCBIname,taxid,strain
0,OR687713,Formosa sejongensis,3081755,PL04
1,JAWQUN000000000,Formosa sejongensis,3081755,PL04
2,OR742320,Polaromonas potterensis,3085630,SM01
3,JAWQLX000000000,Polaromonas potterensis,3085630,SM01


### Combine dataframes into one

In [24]:
combine_df=pd.merge(left=pub4_df, right=srcchk_df, left_on='accession', right_on='accession', how = 'outer')
combine_df = combine_df[['PublishedName', 'NCBIname', 'Strains', 'accession', 'strain', 'Basionym', 'Authority', 'taxid', 'DOI', 'filtered_url' ]]

# Ensure PublishedName is string for sorting
combine_df['PublishedName'] = combine_df['PublishedName'].astype(str)

# Sort alphabetically (case-insensitive)
combine_df = combine_df.sort_values(
    by='PublishedName',
    key=lambda col: col.str.lower(),
    na_position='last'
).reset_index(drop=True)

combine_df

,PublishedName,NCBIname,Strains,accession,strain,Basionym,Authority,taxid,DOI,filtered_url
0,Formosa sejongensis,Formosa sejongensis,"PL04T, DSM 117045T, KCTC 102051T",JAWQUN000000000,PL04,,Choi et al. 2026,3081755,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
1,Formosa sejongensis,Formosa sejongensis,"PL04T, DSM 117045T, KCTC 102051T",OR687713,PL04,,Choi et al. 2026,3081755,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
2,Polaromonas potterensis,Polaromonas potterensis,"SM01, DSM 116566, KCTC 8096",JAWQLX000000000,SM01,,Choi et al. 2026,3085630,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
3,Polaromonas potterensis,Polaromonas potterensis,"SM01, DSM 116566, KCTC 8096",OR742320,SM01,,Choi et al. 2026,3085630,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150


In [25]:
def highlight_rows(row):
    ijsemvalue = row.loc['PublishedName']
    ncbivalue = row.loc['NCBIname']
    if ijsemvalue != ncbivalue:
        color = '#FFB3BA' # Red
    elif ijsemvalue == ncbivalue:
        color = '#BAFFC9' # Green
    return ['background-color: {}'.format(color) for r in row]

new_df = combine_df.style.apply(highlight_rows, axis=1, subset=['PublishedName', 'NCBIname'])
new_df

,PublishedName,NCBIname,Strains,accession,strain,Basionym,Authority,taxid,DOI,filtered_url
0,Formosa sejongensis,Formosa sejongensis,"PL04T, DSM 117045T, KCTC 102051T",JAWQUN000000000,PL04,,Choi et al. 2026,3081755,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
1,Formosa sejongensis,Formosa sejongensis,"PL04T, DSM 117045T, KCTC 102051T",OR687713,PL04,,Choi et al. 2026,3081755,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
2,Polaromonas potterensis,Polaromonas potterensis,"SM01, DSM 116566, KCTC 8096",JAWQLX000000000,SM01,,Choi et al. 2026,3085630,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150
3,Polaromonas potterensis,Polaromonas potterensis,"SM01, DSM 116566, KCTC 8096",OR742320,SM01,,Choi et al. 2026,3085630,https://doi.org/10.1099/ijsem.0.007150,http://dx.doi.org/10.1099/ijsem.0.007150


### write output to excel

In [26]:
new_df.to_excel(output, engine='xlsxwriter', index = False, na_rep = '') 

In [ ]:
combine_df.dtypes